# Dashboard interactivo - Consecuencias de la IA

Cuadernillo construido a partir de la estructura de `ejemplo_dashboard.ipynb` y del boceto `dise?o_diagrama.png`.

La interfaz se organiza en tres pesta?as principales:

- **GRÁFICOS**: visualizaciones interactivas de trabajo, percepcion social, educacion y desinformación.
- **MAPAS**: mapas de patentes de IA por a?o, con vista total y vista proporcional.
- **ML**: modelo exploratorio con Random Forest para estimar la probabilidad de que una persona sienta mas preocupación que entusiasmo por la IA, basado en su percepcion y uso.

Ejecutar todas las celdas y usar el panel que aparece al final.

In [1]:
import importlib.util
import subprocess
import sys

required = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
    'geopandas': 'geopandas',
    'sklearn': 'scikit-learn',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Instalando dependencias faltantes:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('Dependencias principales disponibles.')

Dependencias principales disponibles.


In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import PercentFormatter
import ipywidgets as widgets
from IPython.display import display, HTML, Image, clear_output

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'font.size': 10,
})

PALETTE = {
    'blue': '#5b9bd5',
    'cyan': '#4ecdc4',
    'green': '#6abf69',
    'yellow': '#f6c85f',
    'orange': '#ed7d31',
    'red': '#d95f5f',
    'ink': '#1f2937',
    'muted': '#6b7280',
    'line': '#d1d5db',
}

In [3]:
def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'Dataset').exists() and (candidate / 'cuadernillos').exists():
            return candidate.resolve()
    raise FileNotFoundError('No se encontro la carpeta raiz del proyecto con Dataset/ y cuadernillos/.')

ROOT = find_project_root()
DATA_DIR = ROOT / 'Dataset'
MAP_DIR = ROOT / 'graficos_informe' / 'mapas_patentes_ia'

print(f'Raiz del proyecto: {ROOT}')

work_df = pd.read_csv(DATA_DIR / 'Data_set_01.csv')
students_df = pd.read_csv(DATA_DIR / 'Data_set_03.csv')
public_df = pd.read_csv(DATA_DIR / 'Data_set_05.csv')
incidents_df = pd.read_csv(DATA_DIR / 'Data_set_07.csv', parse_dates=['incident_date', 'first_report_date', 'last_report_date'])

patents_raw = pd.read_csv(ROOT / 'patentes_ia_2018_2024.csv')
patents_raw = patents_raw.loc[:, ~patents_raw.columns.str.startswith('Unnamed')].copy()
patents_raw['TIME_PERIOD'] = pd.to_numeric(patents_raw['TIME_PERIOD'], errors='coerce')
patents_raw['OBS_VALUE'] = pd.to_numeric(patents_raw['OBS_VALUE'], errors='coerce')
# Filtrar por una unica autoridad y medida para evitar conteo doble/multiple.
# 9P50_3 y PF (Patent Families) es el estandar de la OCDE para medir innovacion global.
patents_filtered = patents_raw.dropna(subset=['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE'])
# Se utiliza 9P50_1 (Familias de patentes IP5 por inventor) y AP (Aplicaciones)
# porque es la unica metrica en el dataset que tiene datos actualizados hasta 2024.
patents_filtered = patents_filtered[(patents_filtered['PATENT_AUTHORITIES'] == '9P50_1') & (patents_filtered['MEASURE'] == 'AP')]
patents_df = (
    patents_filtered.groupby(['REF_AREA', 'TIME_PERIOD'], as_index=False)['OBS_VALUE'].sum()
    .rename(columns={'REF_AREA': 'iso3', 'TIME_PERIOD': 'year', 'OBS_VALUE': 'ai_patents'})
)
patents_df['year'] = patents_df['year'].astype(int)

NE_URL = 'https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip'
world = gpd.read_file(NE_URL).rename(columns={'ISO_A3_EH': 'iso3'})
fixes = {
    'France': 'FRA',
    'Norway': 'NOR',
    'Kosovo': 'XK',
    'Somaliland': 'SOM',
}
for name, iso_code in fixes.items():
    world.loc[world['NAME'].eq(name), 'iso3'] = iso_code

world_population = world[['iso3', 'NAME', 'POP_EST', 'geometry']].copy()
world_population = world_population.rename(columns={'NAME': 'country', 'POP_EST': 'population'})
world_population['population'] = pd.to_numeric(world_population['population'], errors='coerce')
world_population.loc[world_population['population'].le(0), 'population'] = np.nan


print('Datasets cargados:')
print(f'- Trabajo: {work_df.shape[0]:,} filas x {work_df.shape[1]} columnas')
print(f'- Educación: {students_df.shape[0]:,} filas x {students_df.shape[1]} columnas')
print(f'- Percepción publica: {public_df.shape[0]:,} filas x {public_df.shape[1]} columnas')
print(f'- Incidentes AIID: {incidents_df.shape[0]:,} filas x {incidents_df.shape[1]} columnas')
print(f'- Patentes IA: {patents_df.shape[0]:,} filas agregadas')
print(f'- Mapa base con población: {world_population["iso3"].nunique():,} países/territorios')

try:
    ds02_df = pd.read_csv(DATA_DIR / 'Data_set_02.csv', encoding='utf-8-sig')
except:
    ds02_df = pd.read_csv(DATA_DIR / 'Data_set_02.csv', encoding='latin1')
ds02_df = ds02_df.rename(columns={
    "What is your age range?": "age_range",
    "What is your gender?": "gender",
    "What is your education level?": "education",
    "Do you generally trust artificial intelligence (AI)?": "trust_ai",
    "Do you think artificial intelligence (AI) will be generally beneficial or harmful to humanity?": "benefit_ai",
    "I think artificial intelligence (AI) could threaten individual freedoms.": "threat_freedoms",
    "Do you think your own job could be affected by artificial intelligence (AI)?": "job_affected",
    "Do you believe that artificial intelligence (AI) should be limited by ethical rules?": "ethical_rules",
})
ds02_df["trust_ai_es"] = ds02_df["trust_ai"].replace({"I trust it": "Confio en la IA", "I don't trust it": "No confio en la IA", "I'm undecided": "Indeciso"})
ds02_df["benefit_ai_es"] = ds02_df["benefit_ai"].replace({"More beneficial than harmful": "Más beneficiosa que dañina", "More harmful than beneficial": "Más dañina que beneficiosa", "Both beneficial and harmful": "Beneficiosa y dañina", "I have no idea": "No se"})
ds02_df["threat_freedoms_es"] = ds02_df["threat_freedoms"].replace({"Strongly agree": "Muy de acuerdo", "Agree": "De acuerdo", "I'm undecided": "Indeciso", "Disagree": "En desacuerdo", "Strongly disagree": "Muy en desacuerdo"})
ds02_df["ethical_rules_es"] = ds02_df["ethical_rules"].replace({"Strongly agree": "Muy de acuerdo", "Agree": "De acuerdo", "I'm undecided": "Indeciso", "Disagree": "En desacuerdo", "Strongly disagree": "Muy en desacuerdo"})
ds02_df["job_affected_es"] = ds02_df["job_affected"].replace({"I definitely think": "Definitivamente si", "I don't think so": "Creo que no", "I'm undecided": "Indeciso", "I have no idea": "No se"})


Raiz del proyecto: C:\Users\marti\Desktop\electivo\Computacion-cientifica
Datasets cargados:
- Trabajo: 1,500 filas x 21 columnas
- Educacion: 8,000 filas x 26 columnas
- Percepcion publica: 5,410 filas x 106 columnas
- Incidentes AIID: 461 filas x 17 columnas
- Patentes IA: 728 filas agregadas
- Mapa base con poblacion: 176 paises/territorios


In [4]:
MISSING_CODES = {97, 98, 99}
WEIGHT = 'WEIGHT_W152' if 'WEIGHT_W152' in public_df.columns else None

single_labels = {
    'AI_HEARD_W152': {1: 'Mucho', 2: 'Un poco', 3: 'Nada'},
    'CNCEXC_W152': {1: 'Más entusiasmo', 2: 'Mas preocupación', 3: 'Ambos por igual'},
    'USEAI_W152': {1: 'Casi constantemente', 2: 'Varias veces al dia', 3: 'Una vez al dia', 4: 'Varias veces por semana', 5: 'Menos seguido'},
    'AICHANGE_W152': {1: 'Muy positivo', 2: 'Algo positivo', 3: 'Positivo y negativo', 4: 'Algo negativo', 5: 'Muy negativo', 6: 'No está seguro'},
    'AIJOBS_W152': {1: 'Más empleos', 2: 'Menos empleos', 3: 'No cambiara mucho', 4: 'No está seguro'},
    'TRSTAIPRS_W152': {1: 'Sí', 2: 'No', 3: 'No está seguro'},
    'AIREG_W152': {1: 'Regulará demasiado', 2: 'No regulará lo suficiente', 3: 'No está seguro'},
    'CHATUSE_W152': {1: 'Sí', 2: 'No'},
}

concern_items = {
    'AICONCERN_a_W152': 'Sesgo en decisiones de IA',
    'AICONCERN_b_W152': 'Suplantación de personas',
    'AICONCERN_c_W152': 'Mal uso de información personal',
    'AICONCERN_d_W152': 'Informacion inexacta',
    'AICONCERN_e_W152': 'No entender que puede hacer la IA',
    'AICONCERN_f_W152': 'Menor conexión entre personas',
    'AICONCERN_g_W152': 'Pérdida de empleos',
}

def clean_codes(series):
    values = pd.to_numeric(series, errors='coerce')
    return values.mask(values.isin(MISSING_CODES))

def weighted_pct(df, col, labels=None, denominator=None):
    values = clean_codes(df[col])
    mask = values.notna()
    if denominator is not None:
        mask &= denominator
    weights = df.loc[mask, WEIGHT] if WEIGHT else pd.Series(1, index=df.loc[mask].index)
    grouped = pd.DataFrame({'value': values.loc[mask], 'weight': weights}).groupby('value')['weight'].sum()
    pct = 100 * grouped / grouped.sum()
    out = pct.reset_index().rename(columns={'value': 'codigo', 'weight': 'porcentaje'})
    if labels:
        out['respuesta'] = out['codigo'].map(labels).fillna(out['codigo'].astype(str))
    else:
        out['respuesta'] = out['codigo'].astype(str)
    return out[['respuesta', 'porcentaje']]

def add_bar_labels(ax, fmt='{:.1f}%'):
    xmax = ax.get_xlim()[1]
    for patch in ax.patches:
        width = patch.get_width()
        if np.isfinite(width):
            ax.text(width + xmax * 0.015, patch.get_y() + patch.get_height()/2, fmt.format(width), va='center', fontsize=9)

future_items = {
    "AIFUTRIMPCT_a_W152": "Atención medica",
    "AIFUTRIMPCT_b_W152": "Educación K-12",
    "AIFUTRIMPCT_c_W152": "Elecciones",
    "AIFUTRIMPCT_d_W152": "Economia",
    "AIFUTRIMPCT_e_W152": "Justicia penal",
    "AIFUTRIMPCT_f_W152": "Arte y entretenimiento",
    "AIFUTRIMPCT_g_W152": "Relaciones personales",
    "AIFUTRIMPCT_h_W152": "Como trabaja la gente",
    "AIFUTRIMPCT_i_W152": "Medio ambiente",
    "AIFUTRIMPCT_j_W152": "Privacidad personal",
}

job_items = {
    "AIJOBIMPCT_a_W152": "Abogados",
    "AIJOBIMPCT_b_W152": "Ingenieros de software",
    "AIJOBIMPCT_c_W152": "Cajeros",
    "AIJOBIMPCT_d_W152": "Trabajadores de fabrica",
    "AIJOBIMPCT_e_W152": "Médicos",
    "AIJOBIMPCT_f_W152": "Docentes",
    "AIJOBIMPCT_g_W152": "Periodistas",
    "AIJOBIMPCT_h_W152": "Conductores de camion",
    "AIJOBIMPCT_i_W152": "Músicos",
    "AIJOBIMPCT_j_W152": "Terapeutas de salud mental",
}

human_items = {
    "HUMANVAI_a_W152": "Diagnóstico medico",
    "HUMANVAI_b_W152": "Escribir una noticia",
    "HUMANVAI_c_W152": "Decidir contratacion",
    "HUMANVAI_d_W152": "Escribir una canción",
    "HUMANVAI_e_W152": "Decidir un préstamo",
    "HUMANVAI_f_W152": "Decidir libertad condicional",
    "HUMANVAI_g_W152": "Conducir a una persona",
    "HUMANVAI_h_W152": "Atención al cliente",
}

def clean_series(s):
    return pd.to_numeric(s, errors="coerce").where(lambda x: ~x.isin(MISSING_CODES))

def weighted_share(df, col, positive_codes, base_mask=None):
    codes = clean_series(df[col])
    mask = codes.notna() & df[WEIGHT].notna()
    if base_mask is not None:
        mask &= base_mask
    denom = df.loc[mask, WEIGHT].sum()
    if denom == 0:
        return np.nan
    return df.loc[mask & codes.isin(positive_codes), WEIGHT].sum() / denom * 100

def stacked_battery_table(items, response_groups, df):
    import pandas as pd
    rows = []
    for col, label in items.items():
        row = {"item": label}
        for group_label, codes in response_groups:
            row[group_label] = weighted_share(df, col, codes)
        rows.append(row)
    return pd.DataFrame(rows)

def plot_stacked_100(ax, table, item_col, response_cols, colors, title):
    import numpy as np
    from matplotlib.ticker import PercentFormatter
    left = np.zeros(len(table))
    y = np.arange(len(table))
    for col, color in zip(response_cols, colors):
        values = table[col].fillna(0).to_numpy()
        ax.barh(y, values, left=left, color=color, label=col)
        left += values
    ax.set_yticks(y)
    ax.set_yticklabels(table[item_col])
    ax.set_xlim(0, 100)
    ax.xaxis.set_major_formatter(PercentFormatter(100))
    ax.set_title(title)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=len(response_cols), frameon=False)


In [5]:
def plot_work_chart(option, role):
    if option == 'Productividad y agotamiento':
        order = work_df.groupby('ai_adoption_stage')['productivity_score'].mean().sort_values().index
        summary = work_df.groupby('ai_adoption_stage')[['productivity_score', 'burnout_score', 'job_satisfaction_1_5']].mean().loc[order]
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
        summary[['productivity_score', 'burnout_score']].plot(kind='barh', ax=axes[0], color=[PALETTE['blue'], PALETTE['orange']])
        axes[0].set_title('Promedios por etapa de adopcion de IA')
        axes[0].set_xlabel('Puntaje promedio')
        axes[0].legend(['Productividad', 'Agotamiento'], frameon=False)
        axes[0].grid(axis='x', alpha=0.22)

        axes[1].barh(summary.index, summary['job_satisfaction_1_5'], color=PALETTE['green'])
        axes[1].set_xlim(0, 5)
        axes[1].set_title('Satisfaccion laboral promedio')
        axes[1].set_xlabel('Escala 1 a 5')
        add_bar_labels(axes[1], '{:.2f}')
        axes[1].grid(axis='x', alpha=0.22)
        plt.show()
    else:
        data = work_df.copy() if role == 'Todas' else work_df[work_df['job_role'].eq(role)].copy()
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        ax.scatter(data['ai_replaces_my_tasks_pct'], data['burnout_score'], s=30, alpha=0.35, color=PALETTE['blue'], edgecolors='none')
        if len(data) > 1:
            m, b = np.polyfit(data['ai_replaces_my_tasks_pct'], data['burnout_score'], 1)
            x_line = np.linspace(data['ai_replaces_my_tasks_pct'].min(), data['ai_replaces_my_tasks_pct'].max(), 100)
            ax.plot(x_line, m * x_line + b, color=PALETTE['red'], linewidth=2.2)
            corr = data['ai_replaces_my_tasks_pct'].corr(data['burnout_score'])
        else:
            corr = np.nan
        ax.set_title(f'Percepción de reemplazo por IA vs agotamiento\n{role}')
        ax.set_xlabel('Tareas percibidas como reemplazables por IA (%)')
        ax.set_ylabel('Puntaje de agotamiento')
        ax.text(0.02, 0.96, f'n = {len(data):,}\nr = {corr:.3f}', transform=ax.transAxes, va='top', bbox=dict(facecolor='white', edgecolor=PALETTE['line'], alpha=0.95))
        ax.grid(alpha=0.24)
        plt.tight_layout()
        plt.show()

def plot_public_chart(option):
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    if option == 'Confianza en IA':
        tab = weighted_pct(public_df, 'TRSTAIPRS_W152', single_labels['TRSTAIPRS_W152'])
        ax.barh(tab['respuesta'], tab['porcentaje'], color=[PALETTE['green'], PALETTE['red'], PALETTE['yellow']])
        ax.set_title('Confianza en que la información de IA es precisa')
    elif option == 'Impacto en empleos':
        tab = weighted_pct(public_df, 'AIJOBS_W152', single_labels['AIJOBS_W152'])
        ax.barh(tab['respuesta'], tab['porcentaje'], color=[PALETTE['green'], PALETTE['red'], PALETTE['blue'], PALETTE['yellow']])
        ax.set_title('Percepción del impacto de IA en el empleo')
    else:
        rows = []
        for col, label in concern_items.items():
            if col in public_df.columns:
                values = clean_codes(public_df[col])
                mask = values.notna()
                weights = public_df.loc[mask, WEIGHT] if WEIGHT else pd.Series(1, index=public_df.loc[mask].index)
                share = 100 * weights.loc[values.loc[mask].isin([1, 2])].sum() / weights.sum()
                rows.append({'preocupación': label, 'porcentaje': share})
        tab = pd.DataFrame(rows).sort_values('porcentaje')
        ax.barh(tab['preocupación'], tab['porcentaje'], color=PALETTE['orange'])
        ax.set_title('Preocupaciónes altas o moderadas asociadas a IA')
    ax.set_xlim(0, 100)
    ax.xaxis.set_major_formatter(PercentFormatter(100))
    ax.grid(axis='x', alpha=0.22)
    add_bar_labels(ax)
    plt.tight_layout()
    plt.show()

def plot_education_chart(option):
    data = students_df.copy()
    data['Uso de IA'] = data['uses_ai'].map({1: 'Usa IA', 0: 'No usa IA'})
    if option == 'Uso IA y rendimiento':
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
        groups = [data.loc[data['Uso de IA'].eq(label), 'final_score'] for label in ['Usa IA', 'No usa IA']]
        axes[0].boxplot(groups, labels=['Usa IA', 'No usa IA'], patch_artist=True, boxprops=dict(facecolor=PALETTE['blue'], alpha=0.65), medianprops=dict(color=PALETTE['ink']))
        axes[0].set_title('Puntaje final según uso de IA')
        axes[0].set_ylabel('Puntaje final')
        perf = data.groupby(['Uso de IA', 'performance_category']).size().unstack(fill_value=0)
        perf_pct = perf.div(perf.sum(axis=1), axis=0) * 100
        perf_pct.reindex(['Usa IA', 'No usa IA'])[['High', 'Medium', 'Low']].plot(kind='bar', stacked=True, ax=axes[1], color=[PALETTE['green'], PALETTE['yellow'], PALETTE['red']])
        axes[1].set_title('Categoría de rendimiento por grupo')
        axes[1].set_ylabel('Porcentaje')
        axes[1].yaxis.set_major_formatter(PercentFormatter(100))
        axes[1].tick_params(axis='x', rotation=0)
        axes[1].legend(title='Rendimiento', frameon=False)
        plt.show()

def incident_terms_count(df):
    etiquetas = {
        'fake news': 'noticias falsas',
        'misinformation': 'información erronea',
        'disinformation': 'desinformación',
        'deepfake': 'deepfake',
        'deep fake': 'deepfake',
        'synthetic media': 'medios sinteticos',
        'manipulated media': 'medios manipulados',
        'false information': 'noticias falsas',
        'false claim': 'afirmaciones falsas',
        'false claims': 'afirmaciones falsas',
        'hoax': 'bulo',
        'propaganda': 'propaganda',
        'election misinformation': 'desinformación electoral',
    }
    values = []
    for item in df['matching_terms'].dropna():
        values.extend([etiquetas.get(v.strip(), v.strip()) for v in str(item).split(';') if v.strip()])
    return pd.Series(values).value_counts()

def plot_incidents_chart(option):
    data = incidents_df[incidents_df['year'].between(2018, 2025, inclusive='both')].copy()
    if option == 'Incidentes por año':
        yearly = data.groupby('year').size().reset_index(name='incidentes')
        fig, ax = plt.subplots(figsize=(8.8, 4.8))
        ax.plot(yearly['year'], yearly['incidentes'], marker='o', color=PALETTE['blue'], linewidth=2.2)
        ax.fill_between(yearly['year'], yearly['incidentes'], color=PALETTE['blue'], alpha=0.18)
        ax.set_title('Incidentes AIID sobre desinformación por año')
        ax.set_xlabel('Año')
        ax.set_ylabel('Cantidad de incidentes')
        ax.grid(axis='y', alpha=0.24)
        plt.tight_layout()
        plt.show()
    elif option == 'Dominios fuente':
        domains = []
        for item in data['source_domains'].dropna():
            domains.extend([v.strip() for v in str(item).split(';') if v.strip()])
        domain_series = pd.Series(domains).value_counts().head(10)
        domain_series = domain_series.iloc[::-1]
        fig, ax = plt.subplots(figsize=(8.8, 4.8))
        ax.barh(domain_series.index, domain_series.values, color=PALETTE['blue'])
        ax.set_title('Dominios fuente más frecuentes')
        ax.set_xlabel('Reportes asociados')
        plt.tight_layout()
        plt.show()
    else:
        terms = incident_terms_count(data).head(8)
        fig, ax = plt.subplots(figsize=(8, 5.5))
        
        def my_autopct(pct):
            return ('%1.1f%%' % pct) if pct >= 4.0 else ''
            
        wedges, texts, autotexts = ax.pie(terms.values, autopct=my_autopct, startangle=90, counterclock=False)
        legend_labels = [f'{idx}: {val}' for idx, val in zip(terms.index, terms.values)]
        ax.legend(wedges, legend_labels, loc='center left', bbox_to_anchor=(1, 0.5))
        ax.set_title('Categorías principales en incidentes')
        ax.axis('equal')
        plt.tight_layout()
        plt.show()


def plot_general_chart(option):
    import matplotlib.pyplot as plt
    if option == 'Impacto esperado de la IA en 20 años, por ámbito':
        future_groups = [
            ("Positivo", [1, 2]),
            ("Mixto", [3]),
            ("Negativo", [4, 5]),
            ("No está seguro", [6]),
        ]
        future = stacked_battery_table(future_items, future_groups, public_df)
        future["orden"] = future["Negativo"] - future["Positivo"]
        future = future.sort_values("orden", ascending=True)
        fig, ax = plt.subplots(figsize=(10.5, 6.1))
        plot_stacked_100(
            ax, future, "item", ["Positivo", "Mixto", "Negativo", "No está seguro"],
            ["#2f8f83", "#d6c7a1", "#b64f5a", "#8b8f97"],
            "Impacto esperado de la IA en 20 años, por ámbito\n(distribución ponderada; cada fila suma 100%)"
        )
        plt.tight_layout()
        plt.show()
    elif option == 'Efecto esperado de la IA sobre empleos por ocupación':
        job_groups = [
            ("Más empleos", [1]),
            ("Menos empleos", [2]),
            ("No cambiara mucho", [3]),
            ("No está seguro", [4]),
        ]
        jobs = stacked_battery_table(job_items, job_groups, public_df)
        jobs["orden"] = jobs["Menos empleos"] - jobs["Más empleos"]
        jobs = jobs.sort_values("orden", ascending=True)
        fig, ax = plt.subplots(figsize=(10.5, 6.1))
        plot_stacked_100(
            ax, jobs, "item", ["Más empleos", "No cambiara mucho", "Menos empleos", "No está seguro"],
            ["#2f8f83", "#d6c7a1", "#b64f5a", "#8b8f97"],
            "Efecto esperado de la IA sobre empleos por ocupación\n(distribución ponderada; cada fila suma 100%)"
        )
        plt.tight_layout()
        plt.show()
    elif option == 'Tareas actuales: evaluación de IA frente a personas':
        human_groups = [
            ("IA mejor", [1]),
            ("IA peor", [2]),
            ("Igual", [3]),
            ("No está seguro", [4]),
        ]
        human = stacked_battery_table(human_items, human_groups, public_df)
        human["orden"] = human["IA peor"] - human["IA mejor"]
        human = human.sort_values("orden", ascending=True)
        fig, ax = plt.subplots(figsize=(10.5, 5.8))
        plot_stacked_100(
            ax, human, "item", ["IA mejor", "Igual", "IA peor", "No está seguro"],
            ["#2f8f83", "#d6c7a1", "#b64f5a", "#8b8f97"],
            "Tareas actuales: evaluación de IA frente a personas\n(distribución ponderada; cada fila suma 100%)"
        )
        plt.tight_layout()
        plt.show()


def plot_ds02_percent_bar(series, title, color="#5b9bd5", order=None):
    import seaborn as sns
    import matplotlib.pyplot as plt
    counts = series.value_counts(dropna=True)
    if order is not None:
        counts = counts.reindex(order).dropna()
    pct = (counts / counts.sum() * 100).round(1)
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    sns.barplot(x=pct.values, y=pct.index, color=color, ax=ax)
    for i, v in enumerate(pct.values):
        ax.text(v + 0.5, i, f"{v:.1f}%", va="center")
    ax.set_title(title)
    ax.set_xlabel("Porcentaje")
    ax.set_ylabel("")
    ax.set_xlim(0, max(10, pct.max() * 1.15))
    plt.tight_layout()
    plt.show()

def plot_ds02_chart(option):
    import seaborn as sns
    import matplotlib.pyplot as plt
    import pandas as pd
    if option == 'Confianza general en IA':
        plot_ds02_percent_bar(ds02_df["trust_ai_es"], "Confianza general en IA", order=["Confio en la IA", "Indeciso", "No confio en la IA"])
    elif option == 'Beneficio o daño para la humanidad':
        plot_ds02_percent_bar(ds02_df["benefit_ai_es"], "Beneficio o daño para la humanidad", color="#ed7d31", order=["Más beneficiosa que dañina", "Beneficiosa y dañina", "Más dañina que beneficiosa", "No se"])
    elif option == 'Amenaza a libertades individuales':
        plot_ds02_percent_bar(ds02_df["threat_freedoms_es"], "Amenaza a libertades individuales", color="#5b9bd5", order=["Muy de acuerdo", "De acuerdo", "Indeciso", "En desacuerdo", "Muy en desacuerdo"])
    elif option == 'Apoyo a reglas éticas':
        plot_ds02_percent_bar(ds02_df["ethical_rules_es"], "Apoyo a reglas éticas", color="#a9d18e", order=["Muy de acuerdo", "De acuerdo", "Indeciso", "En desacuerdo", "Muy en desacuerdo"])
    elif option == 'IA podría afectar mi trabajo':
        plot_ds02_percent_bar(ds02_df["job_affected_es"], "IA podría afectar mi trabajo", color="#9dc3e6", order=["Definitivamente si", "Creo que no", "Indeciso", "No se"])



In [6]:
def render_graphics_tab(category, dataset, option):
    clear_output(wait=True)
    display(HTML('<h3 style="margin:0 0 8px 0">GRÁFICOS</h3>'))
    if category == 'Percepción':
        if dataset == 'Pew Research':
            plot_public_chart(option)
        elif dataset == 'Kaggle':
            plot_ds02_chart(option)
    elif category == 'Educación':
        plot_education_chart(option)
    else:
        plot_incidents_chart(option)

category_options = {
    'Percepción': {
        'Pew Research': ['Confianza en IA', 'Impacto en empleos', 'Preocupaciónes'],
        'Kaggle': ['Confianza general en IA', 'Beneficio o daño para la humanidad', 'Amenaza a libertades individuales', 'Apoyo a reglas éticas', 'IA podría afectar mi trabajo']
    },
    'Educación': ['Uso IA y rendimiento'],
    'Desinformación': ['Incidentes por año', 'Categorías', 'Dominios fuente'],
}

category_toggle = widgets.ToggleButtons(options=['Percepción', 'Educación', 'Desinformación'], value='Percepción')
dataset_toggle = widgets.ToggleButtons(options=list(category_options['Percepción'].keys()), value='Pew Research')
graph_dropdown = widgets.Dropdown(options=category_options['Percepción']['Pew Research'], value='Confianza en IA', layout=widgets.Layout(width='400px'))
graphics_output = widgets.Output(layout={'border': '1px solid #d1d5db', 'padding': '10px', 'min_height': '420px', 'margin': '10px 0 0 0', 'background_color': 'white'})

category_label = widgets.HTML("<div style='font-size:14px; font-weight:600; padding-top:4px;'>Eje:</div>", layout=widgets.Layout(width='80px'))
dataset_label = widgets.HTML("<div style='font-size:14px; font-weight:600; padding-top:4px;'>Fuente:</div>", layout=widgets.Layout(width='80px'))
graph_label = widgets.HTML("<div style='font-size:14px; font-weight:600; padding-top:4px;'>Gráfico:</div>", layout=widgets.Layout(width='80px'))

graph_row1 = widgets.HBox([category_label, category_toggle], layout=widgets.Layout(align_items='center', margin='0 0 10px 0'))
graph_row2 = widgets.HBox([dataset_label, dataset_toggle], layout=widgets.Layout(align_items='center', margin='0 0 10px 0'))
graph_row3 = widgets.HBox([graph_label, graph_dropdown], layout=widgets.Layout(align_items='center', margin='0 0 10px 0'))

graphics_panel = widgets.VBox([
    graph_row1,
    graph_row2,
    graph_row3,
    graphics_output,
], layout=widgets.Layout(padding='10px 0 0 0'))

def sync_graph_options(change=None):
    cat = category_toggle.value
    if cat == 'Percepción':
        graphics_panel.children = [graph_row1, graph_row2, graph_row3, graphics_output]
        ds = dataset_toggle.value
        options = category_options[cat][ds]
    else:
        graphics_panel.children = [graph_row1, graph_row3, graphics_output]
        options = category_options[cat]
    
    graph_dropdown.options = options
    graph_dropdown.value = options[0]

category_toggle.observe(sync_graph_options, names='value')
dataset_toggle.observe(sync_graph_options, names='value')

def update_graphics(change=None):
    with graphics_output:
        render_graphics_tab(category_toggle.value, dataset_toggle.value, graph_dropdown.value)

for widget in [category_toggle, dataset_toggle, graph_dropdown]:
    widget.observe(update_graphics, names='value')

sync_graph_options()
with graphics_output:
    render_graphics_tab(category_toggle.value, dataset_toggle.value, graph_dropdown.value)


In [7]:
years = sorted(patents_df['year'].unique())

TOTAL_CMAP = mcolors.LinearSegmentedColormap.from_list(
    'patentes_totales',
    ['#ffffcc', '#ffeda0', '#fed976', '#feb24c', '#fd8d3c', '#e31a1c', '#800026'],
    N=256,
)
PROP_CMAP = mcolors.LinearSegmentedColormap.from_list(
    'patentes_por_población',
    ['#edf8fb', '#b2e2e2', '#66c2a4', '#2ca25f', '#006d2c'],
    N=256,
)

label_offsets = {
    'CHN': (18, -10), 'USA': (-28, 8), 'JPN': (18, 7), 'KOR': (16, -6),
    'DEU': (0, 12), 'FRA': (-18, -10), 'GBR': (-18, 10), 'TWN': (18, -8),
    'CAN': (-24, 14), 'AUS': (18, -10), 'ITA': (-8, -12),
}

def build_patents_world(year):
    year_data = patents_df[patents_df['year'].eq(year)].copy()
    mapped = world_population.merge(year_data[['iso3', 'ai_patents']], on='iso3', how='left')
    mapped['ai_patents'] = mapped['ai_patents'].fillna(0)
    mapped['patentes_por_millón'] = mapped['ai_patents'] / mapped['population'] * 1_000_000
    mapped.loc[~np.isfinite(mapped['patentes_por_millón']), 'patentes_por_millón'] = np.nan
    return mapped

def draw_patents_map(year, mode, image_width=920):
    mapped = build_patents_world(year)
    fig_width = max(10, min(18, image_width / 70))
    fig_height = fig_width * 0.56

    if mode == 'Total':
        value_col = 'ai_patents'
        cmap = TOTAL_CMAP
        title = f'Patentes de Inteligencia Artificial por país - {year}'
        subtitle = 'Vista total: cantidad absoluta de patentes IA registradas por país'
        color_label = 'Patentes IA totales (escala log)'
        table_value = 'Patentes IA totales'
        formatter = lambda value: f'{value:,.1f}'
    else:
        value_col = 'patentes_por_millón'
        cmap = TOTAL_CMAP
        title = f'Patentes de Inteligencia Artificial por población - {year}'
        subtitle = 'Vista proporcional: patentes IA por millón de habitantes de cada país'
        color_label = 'Patentes IA por millón de habitantes'
        table_value = 'Patentes por millón'
        formatter = lambda value: f'{value:.2f}'

    positive = mapped.loc[mapped[value_col].gt(0), value_col]
    vmax = float(positive.quantile(0.95)) if len(positive) else 1.0
    vmax = max(vmax, 0.01)
    if mode == 'Proporcional':
        # Modo proporcional (normalizado por población): escala lineal con percentil 97 como tope.
        # El percentil 97 evita que outliers extremos saturen la paleta de colores.
        norm = mcolors.Normalize(vmin=0, vmax=vmax)
    else:
        # Modo total (absoluto): escala logaritmica para distribuir mejor colores
        # dado el enorme rango entre países grandes (CHN ~100k) y pequenos (~10).
        vmin_log = float(positive.min()) if len(positive) else 1.0
        vmin_log = max(vmin_log, 1.0)  # LogNorm requiere vmin > 0
        norm = mcolors.LogNorm(vmin=vmin_log, vmax=vmax)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height), facecolor='white')
    ax.set_facecolor('white')
    mapped.plot(ax=ax, color='#ffffff', edgecolor='#d1d5db', linewidth=0.35)
    mapped.loc[mapped[value_col].gt(0)].plot(
        ax=ax,
        column=value_col,
        cmap=cmap,
        norm=norm,
        edgecolor='#d1d5db',
        linewidth=0.35,
        legend=False,
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, shrink=0.68)
    cbar.set_label(color_label, fontsize=10, labelpad=10)
    cbar.ax.tick_params(labelsize=8)

    top_labels = mapped.loc[mapped[value_col].gt(0)].nlargest(10, value_col).copy()
    if mode == 'Proporcional':
        key_total_labels = mapped.loc[mapped['iso3'].isin(['CHN', 'USA', 'JPN', 'KOR'])].copy()
        top_labels = (
            pd.concat([top_labels, key_total_labels], ignore_index=True)
            .drop_duplicates(subset='iso3')
        )
    top_labels['point'] = top_labels.geometry.representative_point()
    for _, row in top_labels.iterrows():
        point = row['point']
        dx, dy = label_offsets.get(row['iso3'], (10, 7))
        ax.annotate(
            f"{row['country']}\n{formatter(row[value_col])}",
            xy=(point.x, point.y),
            xytext=(point.x + dx, point.y + dy),
            fontsize=7,
            color='#111827',
            fontweight='bold',
            ha='center',
            va='center',
            arrowprops=dict(arrowstyle='-', color='#64748b', lw=0.7),
            bbox=dict(boxstyle='round,pad=0.28', facecolor='white', edgecolor='#cbd5e1', alpha=0.92),
        )

    ax.set_title(title, fontsize=16, fontweight='bold', color='#111827', pad=26)
    ax.text(0.5, 1.0, subtitle, transform=ax.transAxes, ha='center', va='bottom', fontsize=10, color='#4b5563')
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

    table_cols = ['iso3', 'country', 'ai_patents', 'population']
    if value_col != 'ai_patents':
        table_cols.append(value_col)
    table_source = mapped.loc[mapped[value_col].gt(0)].nlargest(12, value_col).copy()
    if mode == 'Proporcional':
        key_rows = mapped.loc[mapped['iso3'].isin(['CHN', 'USA', 'JPN', 'KOR'])].copy()
        table_source = (
            pd.concat([table_source, key_rows], ignore_index=True)
            .drop_duplicates(subset='iso3')
        )
    table = table_source[table_cols].copy()
    table['ai_patents'] = table['ai_patents'].map(lambda value: f'{value:,.1f}')
    table['population'] = table['population'].map(lambda value: f'{value:,.0f}')
    if value_col != 'ai_patents':
        table[value_col] = table[value_col].map(formatter)
    table = table.rename(columns={
        'iso3': 'País ISO3',
        'country': 'Pais',
        'ai_patents': 'Patentes IA totales',
        'population': 'Población estimada',
        value_col: table_value,
    })
    display(HTML(table.to_html(index=False)))

def render_map_tab(year, mode, image_width):
    clear_output(wait=True)
    display(HTML('<h3 style="margin:0 0 8px 0">MAPAS</h3>'))
    draw_patents_map(year, mode, image_width)

year_dropdown = widgets.Dropdown(options=years, value=2018 if 2018 in years else min(years), description='Año')
map_mode = widgets.ToggleButtons(options=['Total', 'Proporcional'], value='Total', description='Vista')
zoom_slider = widgets.IntSlider(value=920, min=520, max=1300, step=40, description='Zoom', continuous_update=False)
map_output = widgets.Output(layout={'border': '1px solid #d1d5db', 'padding': '10px', 'min_height': '500px'})

def update_map(change=None):
    with map_output:
        render_map_tab(year_dropdown.value, map_mode.value, zoom_slider.value)

for widget in [year_dropdown, map_mode, zoom_slider]:
    widget.observe(update_map, names='value')

with map_output:
    render_map_tab(year_dropdown.value, map_mode.value, zoom_slider.value)

maps_panel = widgets.VBox([
    widgets.HBox([year_dropdown, map_mode, zoom_slider]),
    map_output,
])

In [8]:
def fit_perception_models():
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score
    import pandas as pd
    
    # === MODELO V1 (11 variables) ===
    features_v1 = [
        'AI_HEARD_W152', 'USEAI_W152', 'CHATUSE_W152', 'TRSTAIPRS_W152',
        'AIJOBS_W152', 'AIREG_W152', 'PERSBENHRM_W152',
        'F_AGECAT', 'F_GENDER', 'F_EDUCCAT', 'F_INC_TIER2'
    ]
    target = 'CNCEXC_W152'
    
    df_model = public_df[features_v1 + [target]].copy()
    for col in df_model.columns:
        df_model[col] = clean_codes(df_model[col])
    
    df_model = df_model.dropna()
    X_v1 = df_model[features_v1].astype(float)
    y = (df_model[target] == 2).astype(int)
    
    X_train1, X_test1, y_train1, y_test1 = train_test_split(X_v1, y, test_size=0.20, random_state=42, stratify=y)
    rf_v1 = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
    rf_v1.fit(X_train1, y_train1)
    acc_v1 = accuracy_score(y_test1, rf_v1.predict(X_test1))
    
    # === MODELO V2 (5 variables - Regla del 80%) ===
    features_v2 = [
        'PERSBENHRM_W152', 'TRSTAIPRS_W152', 'CHATUSE_W152', 'AIJOBS_W152', 'USEAI_W152'
    ]
    # V2 usa su propio subset: solo descarta filas con NaN en sus 5 variables,
    # no hereda el filtro mas estricto de V1 (11 variables).
    df_model_v2 = public_df[features_v2 + [target]].copy()
    for col in df_model_v2.columns:
        df_model_v2[col] = clean_codes(df_model_v2[col])
    df_model_v2 = df_model_v2.dropna()
    X_v2 = df_model_v2[features_v2].astype(float)
    y_v2 = (df_model_v2[target] == 2).astype(int)
    X_train2, X_test2, y_train2, y_test2 = train_test_split(X_v2, y_v2, test_size=0.20, random_state=42, stratify=y_v2)
    
    rf_v2 = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
    rf_v2.fit(X_train2, y_train2)
    acc_v2 = accuracy_score(y_test2, rf_v2.predict(X_test2))
    
    return features_v1, rf_v1, acc_v1, features_v2, rf_v2, acc_v2

feat_v1, rf_v1, acc_v1, feat_v2, rf_v2, acc_v2 = fit_perception_models()

def predict_perception_v1(values):
    import pandas as pd
    row = pd.DataFrame([values])
    prob = rf_v1.predict_proba(row)[0][1] * 100
    return prob

def predict_perception_v2(values):
    import pandas as pd
    row = pd.DataFrame([values])
    prob = rf_v2.predict_proba(row)[0][1] * 100
    return prob

# MAPA DE VALORES COMÚN
maps = {
    'AI_HEARD_W152': {'Mucho': 1, 'Un poco': 2, 'Nada': 3},
    'USEAI_W152': {'Constantemente': 1, 'Varias veces al dia': 2, 'Una vez al dia': 3, 'Varias veces por semana': 4, 'Rara vez': 5},
    'CHATUSE_W152': {'Sí usa ChatGPT': 1, 'No usa ChatGPT': 2},
    'TRSTAIPRS_W152': {'Sí': 1, 'No': 2, 'No está seguro': 3},
    'AIJOBS_W152': {'Más empleos': 1, 'Menos empleos': 2, 'No cambiara mucho': 3, 'No sabe': 4},
    'AIREG_W152': {'Regulará demasiado': 1, 'No regulará lo suficiente': 2, 'No está seguro': 3},
    'PERSBENHRM_W152': {'Mas daños': 1, 'Equilibrado': 3, 'Más beneficios': 2},
    'F_AGECAT': {'18-29': 1, '30-49': 2, '50-64': 3, '65+': 4},
    'F_GENDER': {'Hombre': 1, 'Mujer': 2, 'Otro': 3},
    'F_EDUCCAT': {'Sin bachillerato': 1, 'Bachillerato': 2, 'Universidad+': 3},
    'F_INC_TIER2': {'Bajos': 1, 'Medios': 2, 'Altos': 3}
}

# ----------------- TABS V1 -----------------
def render_ml_tab_v1(ai_heard, use_ai, chat_use, trust_ai, ai_jobs, ai_reg, pers_harm, age_cat, gender, educ, income):
    clear_output(wait=True)
    values = {
        'AI_HEARD_W152': maps['AI_HEARD_W152'][ai_heard],
        'USEAI_W152': maps['USEAI_W152'][use_ai],
        'CHATUSE_W152': maps['CHATUSE_W152'][chat_use],
        'TRSTAIPRS_W152': maps['TRSTAIPRS_W152'][trust_ai],
        'AIJOBS_W152': maps['AIJOBS_W152'][ai_jobs],
        'AIREG_W152': maps['AIREG_W152'][ai_reg],
        'PERSBENHRM_W152': maps['PERSBENHRM_W152'][pers_harm],
        'F_AGECAT': maps['F_AGECAT'][age_cat],
        'F_GENDER': maps['F_GENDER'][gender],
        'F_EDUCCAT': maps['F_EDUCCAT'][educ],
        'F_INC_TIER2': maps['F_INC_TIER2'][income]
    }
    prob = predict_perception_v1(values)
    category = 'Alta preocupación' if prob >= 50 else 'Mayor entusiasmo / Ambos'
    color = PALETTE['red'] if prob >= 50 else PALETTE['blue']
    display(HTML('<h3 style="margin:0 0 8px 0">PREDICCION INTERACTIVA V1 (11 variables)</h3>'))
    display(HTML(f"""
    <div style="font-family:Arial; border:1px solid #d1d5db; padding:20px; max-width:800px; background-color:#f9fafb; border-radius:8px;">
      <div style="font-size:16px; color:#1f2937; margin-bottom:10px;">Probabilidad de tener mas preocupación que entusiasmo:</div>
      <div style="width: 100%; background-color: #e5e7eb; border-radius: 9999px; height: 28px; position: relative;">
        <div style="background-color: {color}; height: 28px; border-radius: 9999px; width: {prob}%; transition: width 0.5s ease-in-out;"></div>
        <div style="position: absolute; top: 3px; left: 15px; color: {'white' if prob > 15 else '#1f2937'}; font-weight: bold; font-size: 15px;">{prob:.1f}%</div>
      </div>
      <div style="font-size:22px; font-weight:700; color:{color}; margin-top:15px; text-align:center;">{category}</div>
      <div style="font-size:12px; color:#6b7280; margin-top:10px; text-align:right;">Test Accuracy del Modelo V1: {acc_v1*100:.1f}%</div>
    </div>
    """))

ml_controls_v1 = {
    'pers_harm': widgets.Dropdown(options=['Más beneficios', 'Equilibrado', 'Mas daños'], value='Equilibrado', description='Balance:'),
    'ai_jobs': widgets.Dropdown(options=['Más empleos', 'Menos empleos', 'No cambiara mucho', 'No sabe'], value='Menos empleos', description='Empleo:'),
    'ai_heard': widgets.Dropdown(options=['Mucho', 'Un poco', 'Nada'], value='Un poco', description='Escucha IA:'),
    'use_ai': widgets.Dropdown(options=['Constantemente', 'Varias veces al dia', 'Una vez al dia', 'Varias veces por semana', 'Rara vez'], value='Varias veces por semana', description='Uso IA:'),
    'chat_use': widgets.ToggleButtons(options=['Sí usa ChatGPT', 'No usa ChatGPT'], value='Sí usa ChatGPT', description='ChatGPT:'),
    'trust_ai': widgets.Dropdown(options=['Sí', 'No', 'No está seguro'], value='No está seguro', description='Confianza:'),
    'ai_reg': widgets.Dropdown(options=['Regulará demasiado', 'No regulará lo suficiente', 'No está seguro'], value='No está seguro', description='Regulacion:'),
    'age_cat': widgets.Dropdown(options=['18-29', '30-49', '50-64', '65+'], value='30-49', description='Edad:'),
    'gender': widgets.Dropdown(options=['Hombre', 'Mujer', 'Otro'], value='Hombre', description='Genero:'),
    'educ': widgets.Dropdown(options=['Sin bachillerato', 'Bachillerato', 'Universidad+'], value='Universidad+', description='Educación:'),
    'income': widgets.Dropdown(options=['Bajos', 'Medios', 'Altos'], value='Medios', description='Ingresos:'),
}
ml_output_v1 = widgets.Output(layout={'border': '1px solid #d1d5db', 'padding': '15px', 'min_height': '220px', 'background_color': 'white'})
def update_ml_v1(change=None):
    with ml_output_v1:
        render_ml_tab_v1(**{k: v.value for k, v in ml_controls_v1.items()})
for widget in ml_controls_v1.values():
    widget.observe(update_ml_v1, names='value')
with ml_output_v1:
    update_ml_v1()

names_es = {
    'AI_HEARD_W152': 'Escuchó sobre IA', 'USEAI_W152': 'Frecuencia uso IA', 'CHATUSE_W152': 'Uso ChatGPT',
    'TRSTAIPRS_W152': 'Confianza info IA', 'AIJOBS_W152': 'IA y empleos', 'AIREG_W152': 'Regulacion IA',
    'PERSBENHRM_W152': 'Balance beneficio/daño', 'F_AGECAT': 'Edad', 'F_GENDER': 'Genero',
    'F_EDUCCAT': 'Educación', 'F_INC_TIER2': 'Ingresos'
}
static_output_v1 = widgets.Output()
with static_output_v1:
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    importances = pd.Series(rf_v1.feature_importances_, index=feat_v1)
    importances.index = importances.index.map(names_es)
    importances = importances.sort_values(ascending=True)
    colors = [PALETTE['red'] if v >= importances.quantile(0.8) else PALETTE['cyan'] for v in importances]
    bars = ax.barh(importances.index, importances.values, color=colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, importances.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
    ax.set_title('Importancia de las variables (Modelo V1 - 11 variables)', pad=15)
    ax.set_xlabel('Importancia relativa')
    ax.set_xlim(0, importances.max() * 1.2)
    ax.grid(axis='x', alpha=0.2)
    plt.tight_layout()
    plt.show()

row1_v1 = widgets.HBox([ml_controls_v1['pers_harm'], ml_controls_v1['ai_jobs'], ml_controls_v1['trust_ai']])
row2_v1 = widgets.HBox([ml_controls_v1['ai_heard'], ml_controls_v1['use_ai'], ml_controls_v1['ai_reg']])
row3_v1 = widgets.HBox([ml_controls_v1['age_cat'], ml_controls_v1['educ'], ml_controls_v1['income']])
row4_v1 = widgets.HBox([ml_controls_v1['chat_use'], ml_controls_v1['gender']])
ml_panel_v1 = widgets.VBox([
    widgets.HTML("<b>Percepción y uso:</b>"), row1_v1, row2_v1, 
    widgets.HTML("<b style='margin-top:10px; display:block'>Demografia y extras:</b>"), row3_v1, row4_v1,
    widgets.HTML("<br>"), ml_output_v1, widgets.HTML("<hr style='margin-top:20px; margin-bottom:10px;'>"), static_output_v1
])

# ----------------- TABS V2 -----------------
def render_ml_tab_v2(pers_harm, trust_ai, chat_use, ai_jobs, use_ai):
    clear_output(wait=True)
    values = {
        'PERSBENHRM_W152': maps['PERSBENHRM_W152'][pers_harm],
        'TRSTAIPRS_W152': maps['TRSTAIPRS_W152'][trust_ai],
        'CHATUSE_W152': maps['CHATUSE_W152'][chat_use],
        'AIJOBS_W152': maps['AIJOBS_W152'][ai_jobs],
        'USEAI_W152': maps['USEAI_W152'][use_ai],
    }
    prob = predict_perception_v2(values)
    category = 'Alta preocupación' if prob >= 50 else 'Mayor entusiasmo / Ambos'
    color = PALETTE['red'] if prob >= 50 else PALETTE['blue']
    display(HTML('<h3 style="margin:0 0 8px 0">PREDICCION INTERACTIVA V2 (5 variables - 80%)</h3>'))
    display(HTML(f"""
    <div style="font-family:Arial; border:1px solid #d1d5db; padding:20px; max-width:800px; background-color:#f9fafb; border-radius:8px;">
      <div style="font-size:16px; color:#1f2937; margin-bottom:10px;">Probabilidad de tener mas preocupación que entusiasmo:</div>
      <div style="width: 100%; background-color: #e5e7eb; border-radius: 9999px; height: 28px; position: relative;">
        <div style="background-color: {color}; height: 28px; border-radius: 9999px; width: {prob}%; transition: width 0.5s ease-in-out;"></div>
        <div style="position: absolute; top: 3px; left: 15px; color: {'white' if prob > 15 else '#1f2937'}; font-weight: bold; font-size: 15px;">{prob:.1f}%</div>
      </div>
      <div style="font-size:22px; font-weight:700; color:{color}; margin-top:15px; text-align:center;">{category}</div>
      <div style="font-size:12px; color:#6b7280; margin-top:10px; text-align:right;">Test Accuracy del Modelo V2: {acc_v2*100:.1f}%</div>
    </div>
    """))

ml_controls_v2 = {
    'pers_harm': widgets.Dropdown(options=['Más beneficios', 'Equilibrado', 'Mas daños'], value='Equilibrado', description='Balance:'),
    'trust_ai': widgets.Dropdown(options=['Sí', 'No', 'No está seguro'], value='No está seguro', description='Confianza:'),
    'chat_use': widgets.ToggleButtons(options=['Sí usa ChatGPT', 'No usa ChatGPT'], value='Sí usa ChatGPT', description='ChatGPT:'),
    'ai_jobs': widgets.Dropdown(options=['Más empleos', 'Menos empleos', 'No cambiara mucho', 'No sabe'], value='Menos empleos', description='Empleo:'),
    'use_ai': widgets.Dropdown(options=['Constantemente', 'Varias veces al dia', 'Una vez al dia', 'Varias veces por semana', 'Rara vez'], value='Varias veces por semana', description='Uso IA:'),
}
ml_output_v2 = widgets.Output(layout={'border': '1px solid #d1d5db', 'padding': '15px', 'min_height': '220px', 'background_color': 'white'})
def update_ml_v2(change=None):
    with ml_output_v2:
        render_ml_tab_v2(**{k: v.value for k, v in ml_controls_v2.items()})
for widget in ml_controls_v2.values():
    widget.observe(update_ml_v2, names='value')
with ml_output_v2:
    update_ml_v2()

static_output_v2 = widgets.Output()
with static_output_v2:
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    importances_v2 = pd.Series(rf_v2.feature_importances_, index=feat_v2)
    importances_v2.index = importances_v2.index.map(names_es)
    importances_v2 = importances_v2.sort_values(ascending=True)
    bars = ax.barh(importances_v2.index, importances_v2.values, color=PALETTE['green'], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, importances_v2.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
    ax.set_title('Importancia de las variables (Modelo V2 - 5 variables, Regla del 80%)', pad=15)
    ax.set_xlabel('Importancia relativa')
    ax.set_xlim(0, importances_v2.max() * 1.2)
    ax.grid(axis='x', alpha=0.2)
    plt.tight_layout()
    plt.show()

row1_v2 = widgets.HBox([ml_controls_v2['pers_harm'], ml_controls_v2['ai_jobs'], ml_controls_v2['trust_ai']])
row2_v2 = widgets.HBox([ml_controls_v2['use_ai'], ml_controls_v2['chat_use']])
ml_panel_v2 = widgets.VBox([
    widgets.HTML("<b>Percepción y uso (variables reducidas):</b>"), row1_v2, row2_v2, 
    widgets.HTML("<br>"), ml_output_v2, widgets.HTML("<hr style='margin-top:20px; margin-bottom:10px;'>"), static_output_v2
])


In [ ]:
header = widgets.HTML("""
<style>
.dashboard-shell {font-family: Arial, sans-serif; border: 1px solid #cbd5e1; padding: 12px; background: #ffffff;}
.dashboard-title {font-size: 22px; font-weight: 700; color: #111827; margin-bottom: 4px;}
.dashboard-subtitle {font-size: 13px; color: #6b7280; margin-bottom: 10px;}
.widget-tab > .p-TabBar .p-TabBar-tab {font-weight: 700;}
</style>
<div class="dashboard-title">Dashboard IA: trabajo, educacion, percepcion y territorio</div>
<div class="dashboard-subtitle">&nbsp;</div>
""")

general_options = [
    'Impacto esperado de la IA en 20 años, por ámbito',
    'Efecto esperado de la IA sobre empleos por ocupación',
    'Tareas actuales: evaluación de IA frente a personas'
]
general_dropdown = widgets.Dropdown(options=general_options, value=general_options[0], description='Gráfico', layout=widgets.Layout(width='600px'))
general_output = widgets.Output(layout={'border': '1px solid #d1d5db', 'padding': '10px', 'min_height': '420px'})

def update_general(change=None):
    with general_output:
        clear_output(wait=True)
        display(HTML('<h3 style="margin:0 0 8px 0">GENERAL</h3>'))
        plot_general_chart(general_dropdown.value)

general_dropdown.observe(update_general, names='value')
with general_output:
    clear_output(wait=True)
    display(HTML('<h3 style="margin:0 0 8px 0">GENERAL</h3>'))
    plot_general_chart(general_dropdown.value)

general_panel = widgets.VBox([
    widgets.HBox([general_dropdown]),
    general_output,
])

main_tabs = widgets.Tab(children=[graphics_panel, general_panel, maps_panel, ml_panel_v1, ml_panel_v2])
main_tabs.set_title(0, 'GRÁFICOS')
main_tabs.set_title(1, 'GENERAL')
main_tabs.set_title(2, 'MAPAS')
main_tabs.set_title(3, 'ML V1 (Original)')
main_tabs.set_title(4, 'ML V2 (80%)')

dashboard = widgets.VBox([header, main_tabs], layout=widgets.Layout(border='1px solid #cbd5e1', padding='12px'))
display(dashboard)
